In [1]:

import pandas as pd                  # to read CSV files
import mysql.connector               # to connect to MySQL
from datetime import datetime        # to work with dates

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
# Connect Python to your MySQL database
conn = mysql.connector.connect(
    host="localhost",      # "localhost" means your own laptop
    user="root",           # MySQL username
    password="root@425",   # your MySQL password
    database="fakenews_dw" # the database we created
)

# cursor is like a pen that writes SQL commands into MySQL
cursor = conn.cursor()

print("Connected to MySQL successfully!")

Connected to MySQL successfully!


In [3]:
fake = pd.read_csv('Fake.csv')
real = pd.read_csv('True.csv')

print(f"Fake articles loaded: {len(fake)}")
print(f"Real articles loaded: {len(real)}")

# TRANSFORM — Add label column
# 0 = Fake, 1 = Real
fake['label'] = 0
real['label'] = 1

# Combine both into one big dataframe
df = pd.concat([fake, real], ignore_index=True)

# Shuffle so Fake and Real are mixed
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nTotal articles combined: {len(df)}")
print(f"\nFirst 3 rows preview:")
print(df[['title', 'subject', 'label']].head(3))

Fake articles loaded: 23481
Real articles loaded: 21417

Total articles combined: 44898

First 3 rows preview:
                                               title       subject  label
0  Ben Stein Calls Out 9th Circuit Court: Committ...       US_News      0
1  Trump drops Steve Bannon from National Securit...  politicsNews      1
2  Puerto Rico expects U.S. to lift Jones Act shi...  politicsNews      1


In [4]:
categories = df['subject'].unique()
print(f"Categories found: {categories}")

# Insert each category into DimCategory table
for cat in categories:
    cursor.execute(
        "INSERT IGNORE INTO DimCategory (category_name) VALUES (%s)", 
        (cat,)
    )
conn.commit()
print("\nCategories loaded into MySQL!")

# Create a map: category_name → category_id
# We need this later when inserting articles
cursor.execute("SELECT category_id, category_name FROM DimCategory")
cat_map = {name: cid for cid, name in cursor.fetchall()}
print(f"\nCategory map: {cat_map}")

Categories found: ['US_News' 'politicsNews' 'News' 'Government News' 'left-news' 'worldnews'
 'politics' 'Middle-east']

Categories loaded into MySQL!

Category map: {'US_News': 1, 'politicsNews': 2, 'News': 3, 'Government News': 4, 'left-news': 5, 'worldnews': 6, 'politics': 7, 'Middle-east': 8}


In [6]:
# Get all unique dates from dataset
unique_dates = df['date'].unique()
print(f"Total unique dates found: {len(unique_dates)}")

# Insert each date into DimDate table
inserted = 0
skipped = 0

for d in unique_dates:
    try:
        # Skip if it looks like a URL or is too long
        if str(d).startswith('http') or len(str(d)) > 50:
            skipped += 1
            continue
            
        # Try to parse the date properly
        parsed = pd.to_datetime(d)
        cursor.execute(
            "INSERT INTO DimDate (full_date, month, year) VALUES (%s, %s, %s)",
            (str(d)[:50], int(parsed.month), int(parsed.year))
        )
        inserted += 1
    except:
        # If date format is weird, store with 0s
        cursor.execute(
            "INSERT INTO DimDate (full_date, month, year) VALUES (%s, %s, %s)",
            (str(d)[:50], 0, 0)
        )
        inserted += 1

conn.commit()
print(f"Dates loaded: {inserted}")
print(f"Skipped (URLs/dirty data): {skipped}")

# Create date map: full_date → date_id
cursor.execute("SELECT date_id, full_date FROM DimDate")
date_map = {full: did for did, full in cursor.fetchall()}
print(f"Total dates in map: {len(date_map)}")

Total unique dates found: 2397
Dates loaded: 2391
Skipped (URLs/dirty data): 6
Total dates in map: 2391


In [7]:
# LOAD — Insert articles into FactNews
print("Starting to load articles... please wait!")

inserted = 0
skipped = 0

for _, row in df.head(5000).iterrows():
    try:
        # Get category_id from our map
        cat_id = cat_map.get(row['subject'], 1)
        
        # Get date_id from our map
        date_id = date_map.get(row['date'], 1)
        
        # Get label (0=Fake, 1=Real)
        label_id = int(row['label'])
        
        # Insert into FactNews
        cursor.execute("""
            INSERT INTO FactNews 
            (title, text_content, category_id, label_id, date_id)
            VALUES (%s, %s, %s, %s, %s)
        """, (
            str(row['title'])[:500],    # limit title to 500 chars
            str(row['text'])[:2000],    # limit text to 2000 chars
            cat_id,
            label_id,
            date_id
        ))
        inserted += 1
        
        # Show progress every 1000 rows
        if inserted % 1000 == 0:
            print(f"  {inserted} articles loaded so far...")
            
    except Exception as e:
        skipped += 1

conn.commit()
print(f"\n✅ Done! {inserted} articles loaded into FactNews!")
print(f"⚠️ Skipped: {skipped} articles")

Starting to load articles... please wait!
  1000 articles loaded so far...
  2000 articles loaded so far...
  3000 articles loaded so far...
  4000 articles loaded so far...
  5000 articles loaded so far...

✅ Done! 5000 articles loaded into FactNews!
⚠️ Skipped: 0 articles
